Celda 1: Configuración y Carga Diferida (Anti-Crash)
Usaremos scan_parquet tanto para Silver como para Gold. Al agrupar y sumar antes de hacer el .collect(), Polars reducirá los 30 millones de filas a solo un par de docenas de filas en memoria, haciendo que este proceso consuma casi 0 de RA

In [1]:
import polars as pl
from pathlib import Path

# Definición de rutas
RUTA_SILVER = Path("../data/02_silver/mef_final_silver.parquet")
RUTA_GOLD_FACT = Path("../data/03_gold/fact_presupuesto.parquet")

print("🔍 Iniciando el proceso de Auditoría y Aseguramiento de Calidad (QA)...")

# Carga diferida (Lazy)
lazy_silver = pl.scan_parquet(RUTA_SILVER)
lazy_gold_fact = pl.scan_parquet(RUTA_GOLD_FACT)

🔍 Iniciando el proceso de Auditoría y Aseguramiento de Calidad (QA)...


Celda 2: Prueba 1 - Conciliación Financiera Global
Aquí sumaremos absolutamente todo el dinero registrado en la capa Silver y lo compararemos contra la capa Gold. El resultado de la resta (Delta) debería ser exactamente 0.0.

In [2]:
print("📊 Control 1: Verificación del Monto Financiero Total Global")

# Sumamos todos los montos en Silver (filtrando nulos para ser justos)
monto_total_silver = (
    lazy_silver
    .filter(pl.col("monto").is_not_null())
    .select(pl.col("monto").sum())
    .collect()
    .item()
)

# Sumamos todos los montos en Gold Fact
monto_total_gold = (
    lazy_gold_fact
    .select(pl.col("monto").sum())
    .collect()
    .item()
)

delta_global = monto_total_silver - monto_total_gold

print(True * "=" + "\n" + "  REPORTE DE INTEGRIDAD GLOBAL" + "\n" + True * "=")
print(f"💰 Monto Total en Capa Silver: {monto_total_silver:,.2f}")
print(f"💰 Monto Total en Capa Gold  : {monto_total_gold:,.2f}")
print(f"📉 Diferencia Financiera (Delta): {delta_global:,.2f}")

if abs(delta_global) < 1e-2:  # Tolerancia por decimales float
    print("✅ ¡ÉXITO! No se perdió ni un solo centavo entre Silver y Gold.")
else:
    print("❌ ¡ALERTA! Hay una discrepancia de dinero. Revisar el pipeline.")

📊 Control 1: Verificación del Monto Financiero Total Global
=
  REPORTE DE INTEGRIDAD GLOBAL
=
💰 Monto Total en Capa Silver: 5,373,286,978,810.52
💰 Monto Total en Capa Gold  : 5,373,286,978,810.52
📉 Diferencia Financiera (Delta): 0.00
✅ ¡ÉXITO! No se perdió ni un solo centavo entre Silver y Gold.


Celda 3: Prueba 2 - Conciliación Granular (Por Año y Fase)
A veces el total global coincide por casualidad, pero los datos internos se cruzaron. Para asegurar que todo esté perfecto, agruparemos el presupuesto por Año Ejecución y por Fase del Gasto (Ej. Compromiso, Devengado, Girado) y cruzaremos ambas capas.

In [3]:
print("🧩 Control 2: Conciliación Granular por Año y Fase del Gasto")

# Agrupación en Silver
df_agrupado_silver = (
    lazy_silver
    .filter(pl.col("monto").is_not_null())
    .group_by(["ano_eje", "fase"])
    .agg(pl.col("monto").sum().alias("monto_silver"))
    .collect()
)

# Agrupación en Gold
df_agrupado_gold = (
    lazy_gold_fact
    .group_by(["ano_eje", "fase"])
    .agg(pl.col("monto").sum().alias("monto_gold"))
    .collect()
)

# Cruzamos ambos reportes resumidos mediante un JOIN
df_conciliacion = (
    df_agrupado_silver
    .join(df_agrupado_gold, on=["ano_eje", "fase"], how="full")
    .with_columns(
        # Llenamos nulos con 0 por si una fase no existe en alguna capa
        pl.col("monto_silver").fill_null(0.0),
        pl.col("monto_gold").fill_null(0.0)
    )
    .with_columns(
        (pl.col("monto_silver") - pl.col("monto_gold")).alias("diferencia")
    )
    .sort(["ano_eje", "fase"])
)

# Mostramos los resultados en pantalla
print(df_conciliacion)

# Verificación automática usando .height (compatible con todas las versiones)
errores = df_conciliacion.filter(pl.col("diferencia").abs() > 1e-2).height

if errores == 0:
    print("\n✅ ¡ESPECTACULAR! La cuadratura es 100% exacta en todas las dimensiones de tiempo y fase.")
else:
    print(f"\n❌ ¡CUIDADO! Se encontraron {errores} combinaciones con descuadres financieros.")

🧩 Control 2: Conciliación Granular por Año y Fase del Gasto
shape: (35, 7)
┌─────────┬───────────────┬──────────────┬───────────────┬───────────────┬────────────┬────────────┐
│ ano_eje ┆ fase          ┆ monto_silver ┆ ano_eje_right ┆ fase_right    ┆ monto_gold ┆ diferencia │
│ ---     ┆ ---           ┆ ---          ┆ ---           ┆ ---           ┆ ---        ┆ ---        │
│ i32     ┆ str           ┆ f64          ┆ i32           ┆ str           ┆ f64        ┆ f64        │
╞═════════╪═══════════════╪══════════════╪═══════════════╪═══════════════╪════════════╪════════════╡
│ 2022    ┆ certificado   ┆ 1.4007e11    ┆ 2022          ┆ certificado   ┆ 1.4007e11  ┆ 0.0        │
│ 2022    ┆ comprometido  ┆ 1.3221e11    ┆ 2022          ┆ comprometido  ┆ 1.3221e11  ┆ 0.000031   │
│ 2022    ┆ comprometido_ ┆ 1.3397e11    ┆ 2022          ┆ comprometido_ ┆ 1.3397e11  ┆ -0.000015  │
│         ┆ anual         ┆              ┆               ┆ anual         ┆            ┆            │
│ 2022    ┆ deve

Celda 4: Prueba 3 - Auditoría de Pérdida de Registros (Filas)
Finalmente, documentaremos cuántas filas se redujeron y demostraremos que esa "pérdida" de filas fue intencional (debido a la limpieza de registros nulos).

In [4]:
print("📉 Control 3: Auditoría de Volumetría (Filas Eliminadas por Limpieza)")

filas_silver_total = lazy_silver.select(pl.len()).collect().item()
filas_silver_sin_nulos = lazy_silver.filter(pl.col("monto").is_not_null()).select(pl.len()).collect().item()
filas_gold = lazy_gold_fact.select(pl.len()).collect().item()

filas_eliminadas_monto_nulo = filas_silver_total - filas_silver_sin_nulos
discrepancia_filas = filas_silver_sin_nulos - filas_gold

print(True * "-")
print(f"📋 Filas totales originales (Silver) : {filas_silver_total:,}")
print(f"🗑️  Filas eliminadas por monto NULL   : {filas_eliminadas_monto_nulo:,}")
print(f"📋 Filas esperadas limpias           : {filas_silver_sin_nulos:,}")
print(f"🚀 Filas finales inyectadas (Gold)   : {filas_gold:,}")
print(True * "-")
print(f"🔎 Discrepancia no explicada de filas: {discrepancia_filas}")

if discrepancia_filas == 0:
    print("✅ ¡PERFECTO! La reducción de filas coincide exactamente con los filtros de limpieza aplicados.")
else:
    print("⚠️ Nota: Hay una diferencia de filas. Esto puede pasar si habian llaves nulas o registros duplicados.")

📉 Control 3: Auditoría de Volumetría (Filas Eliminadas por Limpieza)
-
📋 Filas totales originales (Silver) : 29,530,978
🗑️  Filas eliminadas por monto NULL   : 0
📋 Filas esperadas limpias           : 29,530,978
🚀 Filas finales inyectadas (Gold)   : 29,530,978
-
🔎 Discrepancia no explicada de filas: 0
✅ ¡PERFECTO! La reducción de filas coincide exactamente con los filtros de limpieza aplicados.
